In [ ]:
# Cell 1 — Load config
%run /home/jovyan/work/setup/config.py

In [ ]:
# Cell 2 — Req 4.2: $ Volume per brand per month (from Delta)
df = spark.read.format("delta").load(f"{GOLD_PATH}/summary_sales_by_brand_month")
df.orderBy("year", "month", "brand_nm").show(60, truncate=False)

In [ ]:
# Cell 3 — Live query from fact + dims (proves dimensional model works)
spark.read.format("delta").load(f"{GOLD_PATH}/fact_sales").createOrReplaceTempView("fact_sales")
spark.read.format("delta").load(f"{GOLD_PATH}/dim_date").createOrReplaceTempView("dim_date")
spark.read.format("delta").load(f"{GOLD_PATH}/dim_product").createOrReplaceTempView("dim_product")

spark.sql("""
    SELECT d.year, d.month, d.month_name, p.brand_nm,
           SUM(f.dollar_volume) AS total_dollar_volume
    FROM fact_sales f
    JOIN dim_date    d ON f.date_sk    = d.date_sk
    JOIN dim_product p ON f.product_sk = p.product_sk
    GROUP BY d.year, d.month, d.month_name, p.brand_nm
    ORDER BY d.year, d.month, p.brand_nm
""").show(60, truncate=False)